# 11 - Multi-PDF Chroma Index

## Purpose

This notebook creates one shared Chroma vector database for all uploaded PDF files.

The token-split chunks from all PDFs are embedded using OpenAI embeddings and stored in a single vector index. This allows the chatbot to search across multiple PDFs at the same time.

## Input

This notebook reloads all PDFs from:

```text
/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import getpass

openai_api_key = getpass.getpass("Enter your OpenAI API key: ")

if not openai_api_key:
    raise ValueError("OpenAI API key was not entered.")

os.environ["OPENAI_API_KEY"] = openai_api_key

print("OpenAI API key loaded for this session")

In [0]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import TokenTextSplitter

pdf_folder_path = "/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs"

files_info = dbutils.fs.ls(pdf_folder_path)

pdf_files = [
    file_info.path.replace("dbfs:", "")
    for file_info in files_info
    if file_info.path.lower().endswith(".pdf")
]

print("Number of PDF files found:", len(pdf_files))

all_pdf_docs = []

for pdf_file in pdf_files:
    print("\nLoading PDF:")
    print(pdf_file)

    loader = PyPDFLoader(pdf_file)
    docs = loader.load()

    file_name = os.path.basename(pdf_file)

    for doc in docs:
        doc.metadata["source_file"] = file_name
        doc.metadata["source_path"] = pdf_file
        doc.metadata["page_number"] = doc.metadata.get("page")
        doc.metadata["document_type"] = "pdf"

    all_pdf_docs.extend(docs)

print("\nTotal page documents loaded:", len(all_pdf_docs))

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=500,
    chunk_overlap=50
)

multi_pdf_token_docs = token_splitter.split_documents(all_pdf_docs)

for i, doc in enumerate(multi_pdf_token_docs):
    doc.metadata["chunk_id"] = i

print("Token chunks created:", len(multi_pdf_token_docs))

In [0]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("OpenAI embedding model ready")

In [0]:
test_vector = embedding.embed_query("What is Agentic RAG?")

print("Embedding created")
print("Vector length:", len(test_vector))

In [0]:
from langchain_community.vectorstores import Chroma
import shutil
import os

multi_pdf_persist_directory = "/tmp/chroma_multi_pdf_v1"

if os.path.exists(multi_pdf_persist_directory):
    shutil.rmtree(multi_pdf_persist_directory)
    print("Removed existing Chroma directory")

multi_pdf_vectorstore = Chroma.from_documents(
    documents=multi_pdf_token_docs,
    embedding=embedding,
    persist_directory=multi_pdf_persist_directory
)

print("Multi-PDF Chroma vector database created")
print("Persist directory:", multi_pdf_persist_directory)

In [0]:
try:
    multi_pdf_vectorstore.persist()
    print("Chroma vector database persisted")
except Exception as e:
    print("Persist method not required or not available:", e)

In [0]:
multi_pdf_retriever = multi_pdf_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 12,
        "lambda_mult": 0.7
    }
)

print("Multi-PDF MMR retriever created")

In [0]:
question = "What is retrieval augmented generation?"

retrieved_docs = multi_pdf_retriever.invoke(question)

print("Question:", question)
print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nRetrieved Document {i}")
    print("Source file:", doc.metadata.get("source_file"))
    print("Page number:", doc.metadata.get("page_number"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Preview:")
    print(doc.page_content[:900])
    print("-" * 80)

In [0]:
question = "What is multimodal retrieval?"

retrieved_docs = multi_pdf_retriever.invoke(question)

print("Question:", question)
print("Retrieved documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nRetrieved Document {i}")
    print("Source file:", doc.metadata.get("source_file"))
    print("Page number:", doc.metadata.get("page_number"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Preview:")
    print(doc.page_content[:900])
    print("-" * 80)

In [0]:
print("Multi-PDF vector index ready")
print("PDF files indexed:", len(pdf_files))
print("Token chunks indexed:", len(multi_pdf_token_docs))
print("Chroma path:", multi_pdf_persist_directory)